In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

# Projektpfade relativ zum Notebooks-Ordner
repo_root = pathlib.Path().resolve().parent
data_dir = repo_root / "data"
models_dir = repo_root / "models"
features_dir = data_dir / "features"
splits_dir = data_dir / "splits"

models_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)
splits_dir.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_RANGE = range(1, 26)  # Nachbarn 1..25


# k-Nearest Neighbors (kNN) für Plankton-Klassifikation

Dieses Notebook vergleicht kNN-Varianten (Basis, PCA, Random Subspace) auf vorbereiteten Plankton-Features und wählt das beste Modell für den finalen Vergleich.

## 1. Features laden
Vorgefertigte Feature-Vektoren (Train undersampled, Val/Test vollständig) einlesen und Verteilungen ausgeben. Zusätzlich werden die im Preprocessing berechneten PCA-Features aus `knn_features.npz` geladen (kein erneutes PCA-Fitting in diesem Notebook).

In [ ]:
# Lade DT-Features (ohne PCA)
feat_train = np.load(features_dir / "dt_features.npz", allow_pickle=True)
X_train, y_train = feat_train["X_train"], feat_train["y_train"]
X_val, y_val = feat_train["X_val"], feat_train["y_val"]
X_test, y_test = feat_train["X_test"], feat_train["y_test"]

# Lade vorverarbeitete PCA-Features aus Preprocessing
feat_knn = np.load(features_dir / "knn_features.npz", allow_pickle=True)
X_train_pca = feat_knn["X_train"]
X_val_pca   = feat_knn["X_val"]
X_test_pca  = feat_knn["X_test"]

print("Train-Features:", X_train.shape)
print("Validation-Features:", X_val.shape)
print("Test-Features:", X_test.shape)

print("PCA-Train-Features:", X_train_pca.shape)
print("PCA-Validation-Features:", X_val_pca.shape)
print("PCA-Test-Features:", X_test_pca.shape)

# Klassenverteilung prüfen
for name, labels in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = dict(zip(unique, counts))
    print(f"{name} Klassenverteilung: {dist}")

In [ ]:
# Skalieren der Basis-Features (für Distanz-basierte KNN wichtig)
scaler = StandardScaler(with_mean=False)  # Daten sind >=0, effizient für große Vektoren
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print("Skalierte Shapes:", X_train_s.shape, X_val_s.shape, X_test_s.shape)

## 2. Gewichtungen testen
Vergleicht kNN mit `uniform` vs. `distance` auf skalierten Features; beste Gewichtung dient allen Varianten (Basis, PCA, Random Subspace).

In [ ]:
def eval_knn(X_tr, y_tr, X_val_data, n_neighbors, weights):
    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, n_jobs=-1)
    model.fit(X_tr, y_tr)
    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    val_acc = accuracy_score(y_val, model.predict(X_val_data))
    return model, train_acc, val_acc

models_cmp = {}
for w in ["uniform", "distance"]:
    model, train_acc, val_acc = eval_knn(X_train_s, y_train, X_val_s, n_neighbors=5, weights=w)
    models_cmp[w] = {"model": model, "train": train_acc, "val": val_acc}
    print(f"Gewichtung={w:8s} | Train-Acc={train_acc:.4f} | Val-Acc={val_acc:.4f}")

best_weight = max(models_cmp.items(), key=lambda kv: kv[1]["val"])[0]
best_base_model = models_cmp[best_weight]["model"]
print(f"\nBeste Gewichtung laut Val: {best_weight}")

## 3. Nachbarn-Sweep (Basis, skaliert)
Sweep über `n_neighbors` (1–25) auf skalierten DT-Features; wählt bestes k.

In [ ]:
train_scores = []
val_scores = []
models = []

for k in N_RANGE:
    m, tr, va = eval_knn(X_train_s, y_train, X_val_s, k, best_weight)
    train_scores.append(tr)
    val_scores.append(va)
    models.append(m)

best_idx = int(np.argmax(val_scores))
best_k = list(N_RANGE)[best_idx]
best_model = models[best_idx]

print(f"Bestes k nach Val: {best_k} | Val-Acc={val_scores[best_idx]:.4f} | Train-Acc={train_scores[best_idx]:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(list(N_RANGE), val_scores, marker='s', label='Val')
plt.scatter(best_k, val_scores[best_idx], color='red', s=60, label=f'Best k={best_k}')
plt.xlabel('n_neighbors')
plt.ylabel('Accuracy')
plt.title(f'KNN Validation Accuracy vs n_neighbors (weights={best_weight})')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. PCA-Sweep
Sweep über `n_neighbors` (1–25) auf den vorbereiteten PCA-Features; wählt bestes k.

In [ ]:
train_scores_pca = []
val_scores_pca = []
models_pca = []

for k in N_RANGE:
    m, tr, va = eval_knn(X_train_pca, y_train, X_val_pca, k, best_weight)
    train_scores_pca.append(tr)
    val_scores_pca.append(va)
    models_pca.append(m)

best_idx_pca = int(np.argmax(val_scores_pca))
best_k_pca = list(N_RANGE)[best_idx_pca]
best_model_pca = models_pca[best_idx_pca]

print(f"Bestes k (PCA) nach Val: {best_k_pca} | Val-Acc={val_scores_pca[best_idx_pca]:.4f} | Train-Acc={train_scores_pca[best_idx_pca]:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(list(N_RANGE), val_scores_pca, marker='s', label='Val (PCA)')
plt.scatter(best_k_pca, val_scores_pca[best_idx_pca], color='red', s=60, label=f'Best k={best_k_pca}')
plt.xlabel('n_neighbors')
plt.ylabel('Accuracy')
plt.title(f'KNN PCA Validation Accuracy vs n_neighbors (weights={best_weight})')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Random Subspace Sweep
Sweep über `n_neighbors` (1–15) auf zufällig ausgewähltem Feature-Subset der PCA-Daten.

In [ ]:
# Random Subspace: subset zufälliger Features (auf PCA-Daten)
SUBSPACE_SIZE = min(256, X_train_pca.shape[1])
rng = np.random.default_rng(RANDOM_STATE)
sub_idx = rng.choice(X_train_pca.shape[1], size=SUBSPACE_SIZE, replace=False)

X_train_sub = X_train_pca[:, sub_idx]
X_val_sub   = X_val_pca[:, sub_idx]
X_test_sub  = X_test_pca[:, sub_idx]

# Sweep über k (gleiche Gewichtung)
train_scores_sub = []
val_scores_sub = []
models_sub = []

for k in range(1, 16):  # kleinerer Sweep für Subspace
    m, tr, va = eval_knn(X_train_sub, y_train, X_val_sub, k, best_weight)
    train_scores_sub.append(tr)
    val_scores_sub.append(va)
    models_sub.append(m)

best_idx_sub = int(np.argmax(val_scores_sub))
best_k_sub = list(range(1, 16))[best_idx_sub]
best_model_sub = models_sub[best_idx_sub]

print(f"Bestes k (Subspace) nach Val: {best_k_sub} | Val-Acc={val_scores_sub[best_idx_sub]:.4f} | Train-Acc={train_scores_sub[best_idx_sub]:.4f}")

## 6. Modellvergleich
Vergleicht Test-Accuracy der drei Varianten und zeigt die Confusion-Matrix des besten Modells. Basis nutzt skalierte DT-Features, PCA/Subspace nutzen die vorab berechneten PCA-Features aus dem Preprocessing.

In [ ]:
# Test-Performance der drei Varianten
acc_base = accuracy_score(y_test, best_model.predict(X_test_s))
acc_pca  = accuracy_score(y_test, best_model_pca.predict(X_test_pca))
acc_sub  = accuracy_score(y_test, best_model_sub.predict(X_test_sub))

print(f"Test-Accuracy Basis   (k={best_k},  weights={best_weight}): {acc_base:.4f}")
print(f"Test-Accuracy PCA     (k={best_k_pca}, weights={best_weight}): {acc_pca:.4f}")
print(f"Test-Accuracy Subspace(k={best_k_sub}, weights={best_weight}, features={SUBSPACE_SIZE}): {acc_sub:.4f}")

# Confusion Matrix für bestes Gesamtmodell (höchste Test-Acc)
best_name, best_acc, best_cm_pred = max(
    [
        ("Basis", acc_base, best_model.predict(X_test_s)),
        ("PCA", acc_pca, best_model_pca.predict(X_test_pca)),
        ("Subspace", acc_sub, best_model_sub.predict(X_test_sub)),
    ],
    key=lambda t: t[1]
)

cm = confusion_matrix(y_test, best_cm_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - KNN (Best: {best_name})')
plt.tight_layout()
plt.show()

print('\nClassification report (best Variante):')
print(classification_report(y_test, best_cm_pred))

## 7. Fazit
Nach Anwendung der PCA (95 % erklärte Varianz) erzielten der Standard-kNN und die Random-Subspace-Variante eine vergleichbare Test-Accuracy. PCA entfernt bereits Redundanzen und irrelevante Dimensionen, wodurch der Diversitätseffekt der Subspace-Methode gering bleibt.

## 8. Modell speichern

Das beste k-NN Modell wird für den finalen Vergleich in Notebook 06 gespeichert.


In [ ]:
# Wähle das beste Gesamtmodell (höchste Test-Accuracy)
best_overall_idx = np.argmax([acc_base, acc_pca, acc_sub])
best_model_names = ["knn_base_model.pkl", "knn_pca_model.pkl", "knn_subspace_model.pkl"]
best_models_list = [best_model, best_model_pca, best_model_sub]

best_model_file = best_model_names[best_overall_idx]
best_model_final = best_models_list[best_overall_idx]
best_test_acc = [acc_base, acc_pca, acc_sub][best_overall_idx]

model_path = models_dir / best_model_file
with open(model_path, "wb") as f:
    pickle.dump(best_model_final, f)

# Metadaten speichern
variant_names = ["Base (scaled features)", "PCA", "Random Subspace"]
metadata = {
    "model_type": f"k-NN ({variant_names[best_overall_idx]})",
    "best_variant": variant_names[best_overall_idx],
    "n_neighbors": [best_k, best_k_pca, best_k_sub][best_overall_idx],
    "weights": best_weight,
    "test_accuracy": best_test_acc,
    "n_features": [X_train_s.shape[1], X_train_pca.shape[1], X_train_sub.shape[1]][best_overall_idx],
}

metadata_path = models_dir / "knn_metadata.pkl"
with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

print(f"k-NN Modell und Metadaten gespeichert in {models_dir}")
print(f"  - Beste Variante: {variant_names[best_overall_idx]}")
print(f"  - Test Accuracy: {best_test_acc:.4f}")
print(f"  - n_neighbors: {[best_k, best_k_pca, best_k_sub][best_overall_idx]}")
print(f"  - Gewichtung: {best_weight}")
